# Validation Analysis

This notebook evaluates checkpoint translations and visualizes validation quality with CER, WER, BLEU, and attention maps.

In [ ]:
import torch
from datasets import load_dataset
from config import get_config
from dataset import BilingualDataset
from tokenizer import get_or_build_tokenizer
from translate import load_translation_model, encode_source, greedy_decode
from visualization import plot_attention

In [ ]:
config = get_config()
raw = load_dataset(config['datasource'], f"{config['lang_src']}-{config['lang_tgt']}", split='train')
tokenizer_src = get_or_build_tokenizer(config, raw, config['lang_src'])
tokenizer_tgt = get_or_build_tokenizer(config, raw, config['lang_tgt'])
split = int(len(raw) * 0.9)
validation_raw = raw.select(range(split, len(raw)))
validation = BilingualDataset(validation_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
len(validation)

In [ ]:
model, tokenizer_src, tokenizer_tgt, device = load_translation_model(config)
sample = validation[0]
source = sample['encoder_input'].unsqueeze(0).to(device)
source_mask = sample['encoder_mask'].to(device)
with torch.inference_mode():
    output = greedy_decode(model, source, source_mask, tokenizer_tgt, config['seq_len'], device)
prediction = tokenizer_tgt.decode(output.detach().cpu().numpy(), skip_special_tokens=True)
sample['src_text'], sample['tgt_text'], prediction

In [ ]:
from torchmetrics.text import CharErrorRate, WordErrorRate, BLEUScore
references = [sample['tgt_text']]
predictions = [prediction]
metrics = {
    'CER': CharErrorRate()(predictions, references).item(),
    'WER': WordErrorRate()(predictions, references).item(),
    'BLEU': BLEUScore()(predictions, [[references[0]]]).item()
}
metrics

In [ ]:
encoder_output = model.encode(source, source_mask)
decoder_input = torch.tensor([[tokenizer_tgt.token_to_id('[SOS]')]], device=device)
with torch.inference_mode():
    decoder_output = model.decode(encoder_output, source_mask, decoder_input, torch.ones(1, 1, 1, dtype=torch.bool, device=device))
attention = model.decoder.layers[0].cross_attention_block.attention_scores
if attention is not None:
    source_tokens = tokenizer_src.encode(sample['src_text']).tokens
    target_tokens = ['[SOS]']
    plot_attention(attention, source_tokens, target_tokens, title='Encoder-Decoder Attention')